# Save unmapped data for every (split, bureau)

Lighter-weight version of `load_from_snowflake.ipynb`: pulls only the **unmapped** rows from Snowflake `POWER_DB.<bureau>.t0_trade` (asset-projected columns + ZEST_KEY + ARCHIVE_DATE) and saves them as chunked parquets under:

```
payment_processing_research_data/<bureau>/<split>/unmapped/part-NNNNN.parquet
```

Loop order: **train -> valid -> test**, and within each split, **equifax -> experian -> transunion**. So all three bureaus' train data lands before any valid data, etc.

Use this notebook when you just want the raw bureau columns; run `load_from_snowflake.ipynb` if you also need the mapped output.

In [5]:
%pip install --quiet snowflake-connector-python cryptography


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import gc
import warnings
from pathlib import Path

import pandas as pd

from configs import EQUIFAX, EXPERIAN, TRANSUNION, DATA_DIR, unmapped_dir
from helpers import (
    get_conn,
    load_t0_trade,
    save_in_chunks,
)

warnings.filterwarnings('ignore')

print('DATA_DIR =', DATA_DIR)

You have an incompatible version of 'pyarrow' installed (11.0.0), please install a version that adheres to: 'pyarrow>=14.0.1; extra == "pandas"'


DATA_DIR = /home/jag/payment-processor-research/payment_processing_research_data


## Snowflake connection (RSA key auth)

Same workbench-standard auth as `sample_truist_preprocessed.ipynb` and `load_from_snowflake.ipynb` -- username from `$USER@zest.ai`, RSA key from `~/.snowflake/rsa_key.p8`. No password.

In [4]:
# Sanity check the connection before pulling data
with get_conn('EQUIFAX') as _c:
    print(_c.cursor().execute(
        'SELECT current_user(), current_role(), current_warehouse(), current_database()'
    ).fetch_pandas_all())

  CURRENT_USER() CURRENT_ROLE() CURRENT_WAREHOUSE() CURRENT_DATABASE()
0    jag@zest.ai     POWER_ROLE            POWER_WH           POWER_DB


In [5]:
# Query + load helpers (get_raw_features, build_t0_trade_query, load_t0_trade)
# live in helpers.py. load_t0_trade is imported above and used directly in the
# main loop below.
#
# Quick sanity: print the projected column list per bureau so you can confirm
# the asset is being read correctly.
from helpers import get_raw_features

for b in ['equifax', 'experian', 'transunion']:
    raws = get_raw_features(b)
    print(f'{b}: {len(raws)} raw_feature columns -- first 5: {raws[:5]}')

equifax: 27 raw_feature columns -- first 5: ['ACCOUNT_TYPE', 'ACTIVITY_DESIGNATOR', 'BALANCE', 'CLOSED_DATE', 'CREDIT_LIMIT']
experian: 32 raw_feature columns -- first 5: ['ACCOUNT_CONDITION_CODE', 'AMOUNT_1', 'AMOUNT_1_QUALIFIER', 'AMOUNT_2', 'AMOUNT_2_QUALIFIER']
transunion: 23 raw_feature columns -- first 5: ['ACCOUNT_TYPE', 'CLOSED_DATE', 'CLOSED_DATE_INDICATOR', 'CREDIT_LIMIT_AMOUNT', 'CURRENT_BALANCE_AMOUNT']


In [6]:
# save_in_chunks + CHUNK_SIZE live in helpers.py (imported above).
from helpers import CHUNK_SIZE
print(f'CHUNK_SIZE = {CHUNK_SIZE:,} rows per parquet file')

CHUNK_SIZE = 100,000 rows per parquet file


In [7]:
# OUTER loop: splits in order train -> valid -> test.
# INNER loop: equifax -> experian -> transunion.
# So all three bureaus' train data lands first, then all three valid, then all three test.
SPLITS  = ['train', 'valid', 'test']
BUREAUS = [EQUIFAX, EXPERIAN, TRANSUNION]

for split in SPLITS:
    print(f'\n##### SPLIT = {split} #####')
    for cfg in BUREAUS:
        bureau = cfg['bureau']
        print(f'\n=== {bureau}/{split} ===')

        # 1. Pull from Snowflake
        df = load_t0_trade(cfg, split)

        # 2. The asset's DateConverterV2 expects DATE_OF_REQUEST as a column,
        #    and analysis notebooks rely on it. Set it from ARCHIVE_DATE here
        #    so the unmapped parquets are immediately usable.
        df['DATE_OF_REQUEST'] = df['ARCHIVE_DATE']

        # 3. Save chunked under <bureau>/<split>/unmapped/
        out_dir = unmapped_dir(cfg, split)
        n_chunks = save_in_chunks(df, out_dir)
        print(f'[{bureau}/{split}]   saved {n_chunks} chunks -> {out_dir}')

        # 4. Free memory before the next iteration
        del df
        gc.collect()


##### SPLIT = train #####

=== equifax/train ===
[equifax/train] querying POWER_DB.EQUIFAX.t0_trade  (pull=national_1.1, archive_date=2019-03-31)
[equifax/train]   58,130,132 rows x 28 cols
[equifax/train]   saved 582 chunks -> /home/jag/payment-processor-research/payment_processing_research_data/equifax/train/unmapped

=== experian/train ===
[experian/train] querying POWER_DB.EXPERIAN.t0_trade  (pull=national_1.3, archive_date=2019-03-31)
[experian/train]   44,246,273 rows x 33 cols
[experian/train]   saved 443 chunks -> /home/jag/payment-processor-research/payment_processing_research_data/experian/train/unmapped

=== transunion/train ===
[transunion/train] querying POWER_DB.TRANSUNION.t0_trade  (pull=national_3_refresh, archive_date=2019-03-31)
[transunion/train]   53,313,028 rows x 24 cols
[transunion/train]   saved 534 chunks -> /home/jag/payment-processor-research/payment_processing_research_data/transunion/train/unmapped

##### SPLIT = valid #####

=== equifax/valid ===
[equifax

In [8]:
# Verify the chunked output -- one row per (bureau, split) showing chunks + size.
import pyarrow.dataset as ds

rows = []
for split in SPLITS:
    for cfg in BUREAUS:
        d = Path(unmapped_dir(cfg, split))
        parts = sorted(d.glob('part-*.parquet')) if d.exists() else []
        if not parts:
            rows.append({'bureau': cfg['bureau'], 'split': split, 'chunks': 0,
                         'rows': 0, 'size_mb': 0.0, 'dir': str(d)})
            continue
        n_rows = ds.dataset(str(d), format='parquet').count_rows()
        total_mb = sum(p.stat().st_size for p in parts) / (1024 * 1024)
        rows.append({'bureau': cfg['bureau'], 'split': split,
                     'chunks': len(parts), 'rows': n_rows,
                     'size_mb': round(total_mb, 1), 'dir': str(d)})

pd.DataFrame(rows)

,bureau,split,chunks,rows,size_mb,dir
0,equifax,train,582,58130132,1373.7,/home/jag/payment-processor-research/payment_p...
1,experian,train,443,44246273,2203.3,/home/jag/payment-processor-research/payment_p...
2,transunion,train,534,53313028,1274.3,/home/jag/payment-processor-research/payment_p...
3,equifax,valid,596,59566661,1412.3,/home/jag/payment-processor-research/payment_p...
4,experian,valid,452,45186541,2272.1,/home/jag/payment-processor-research/payment_p...
5,transunion,valid,578,57711027,1365.3,/home/jag/payment-processor-research/payment_p...
6,equifax,test,552,55140816,1286.4,/home/jag/payment-processor-research/payment_p...
7,experian,test,445,44470384,2225.4,/home/jag/payment-processor-research/payment_p...
8,transunion,test,445,44440427,1039.0,/home/jag/payment-processor-research/payment_p...
